In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
import re
import time
import random
import warnings
from collections import Counter
import numpy as np, pandas as pd, polars as pl

import torch
import vllm
from vllm import LLM, SamplingParams

import kaggle_evaluation.aimo_2_inference_server

warnings.simplefilter('ignore')
print('PyTorch version:', torch.__version__)
print('vLLM:', vllm.__version__)

PyTorch version: 2.5.1+cu124
vLLM: 0.7.3


In [2]:
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"

In [3]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True
seed_everything(seed=42)

start_time = time.time()
cutoff_time = start_time + (4 * 60 + 45) * 60
cutoff_times = [int(x) for x in np.linspace(cutoff_time, start_time + 60 * 60, 50)]

In [4]:
if os.getenv('KAGGLE_KERNEL_RUN_TYPE') or os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    llm_model_pth = '/kaggle/input/ds_14b_w4a16/transformers/default/1'
else:
    llm_model_pth = '/root/volume/KirillR/QwQ-32B-Preview-AWQ'

MAX_NUM_SEQS = 16
MAX_MODEL_LEN = 8192 * 3 // 2

llm = LLM(
    llm_model_pth,
    kv_cache_dtype="fp8",
    max_num_seqs=MAX_NUM_SEQS,    # Maximum number of sequences per iteration. Default is 256
    max_model_len=MAX_MODEL_LEN,  # Model context length
    trust_remote_code=True,       # Trust remote code (e.g., from HuggingFace) when downloading the model and tokenizer
    tensor_parallel_size=4,       # The number of GPUs to use for distributed execution with tensor parallelism
    gpu_memory_utilization=0.9,  # The ratio (between 0 and 1) of GPU memory to reserve for the model
    seed=42,
)

tokenizer = llm.get_tokenizer()

INFO 03-16 05:17:31 __init__.py:207] Automatically detected platform cuda.
INFO 03-16 05:18:03 config.py:549] This model supports multiple tasks: {'reward', 'classify', 'score', 'generate', 'embed'}. Defaulting to 'generate'.
INFO 03-16 05:18:07 config.py:1096] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor
INFO 03-16 05:18:07 config.py:1382] Defaulting to use mp for distributed inference
INFO 03-16 05:18:07 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='/kaggle/input/ds_14b_w4a16/transformers/default/1', speculative_config=None, tokenizer='/kaggle/input/ds_14b_w4a16/transformers/default/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=12288, download_dir=None, load_format=auto, tensor_parallel_size=4, pip

2025-03-16 05:18:09,073 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend
(VllmWorkerProcess pid=351) 2025-03-16 05:18:09,091 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend
(VllmWorkerProcess pid=359) 2025-03-16 05:18:09,093 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend
(VllmWorkerProcess pid=354) 2025-03-16 05:18:09,096 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend


(VllmWorkerProcess pid=354) (VllmWorkerProcess pid=351) INFO 03-16 05:18:20 utils.py:916] Found nccl from library libnccl.so.2
INFO 03-16 05:18:20 utils.py:916] Found nccl from library libnccl.so.2
INFO 03-16 05:18:20 pynccl.py:69] vLLM is using nccl==2.21.5
INFO 03-16 05:18:20 utils.py:916] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=359) (VllmWorkerProcess pid=354) (VllmWorkerProcess pid=351) INFO 03-16 05:18:20 utils.py:916] Found nccl from library libnccl.so.2
INFO 03-16 05:18:20 pynccl.py:69] vLLM is using nccl==2.21.5
INFO 03-16 05:18:20 pynccl.py:69] vLLM is using nccl==2.21.5
(VllmWorkerProcess pid=359) INFO 03-16 05:18:20 pynccl.py:69] vLLM is using nccl==2.21.5
WARNING 03-16 05:18:21 custom_all_reduce.py:136] Custom allreduce is disabled because it's not supported on more than two PCIe-only GPUs. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(VllmWorkerProcess pid=351) (VllmWorkerProcess pid=354) (VllmWorkerProcess pid=359) WARNIN

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


WARNING 03-16 05:19:06 logger.py:63] Using KV cache scaling factor 1.0 for fp8_e4m3. This may cause accuracy issues. Please make sure k/v_scale scaling factors are available in the fp8 checkpoint.
(VllmWorkerProcess pid=354) (VllmWorkerProcess pid=351) WARNING 03-16 05:19:07 logger.py:63] Using KV cache scaling factor 1.0 for fp8_e4m3. This may cause accuracy issues. Please make sure k/v_scale scaling factors are available in the fp8 checkpoint.
WARNING 03-16 05:19:07 logger.py:63] Using KV cache scaling factor 1.0 for fp8_e4m3. This may cause accuracy issues. Please make sure k/v_scale scaling factors are available in the fp8 checkpoint.
(VllmWorkerProcess pid=359) WARNING 03-16 05:19:07 logger.py:63] Using KV cache scaling factor 1.0 for fp8_e4m3. This may cause accuracy issues. Please make sure k/v_scale scaling factors are available in the fp8 checkpoint.
INFO 03-16 05:19:07 model_runner.py:1115] Loading model weights took 2.3542 GB
(VllmWorkerProcess pid=351) (VllmWorkerProcess pi

Capturing CUDA graph shapes:   0%|          | 0/5 [00:00<?, ?it/s]2025-03-16 05:19:33,482 - INFO - flashinfer.jit: Loading JIT ops: batch_prefill_with_kv_cache_dtype_q_f16_dtype_kv_e4m3_dtype_o_f16_dtype_idx_i32_head_dim_qk_128_head_dim_vo_128_posenc_0_use_swa_False_use_logits_cap_False_f16qk_False
(VllmWorkerProcess pid=354) (VllmWorkerProcess pid=351) (VllmWorkerProcess pid=359) 2025-03-16 05:19:33,482 - INFO - flashinfer.jit: Loading JIT ops: batch_prefill_with_kv_cache_dtype_q_f16_dtype_kv_e4m3_dtype_o_f16_dtype_idx_i32_head_dim_qk_128_head_dim_vo_128_posenc_0_use_swa_False_use_logits_cap_False_f16qk_False
2025-03-16 05:19:33,482 - INFO - flashinfer.jit: Loading JIT ops: batch_prefill_with_kv_cache_dtype_q_f16_dtype_kv_e4m3_dtype_o_f16_dtype_idx_i32_head_dim_qk_128_head_dim_vo_128_posenc_0_use_swa_False_use_logits_cap_False_f16qk_False
2025-03-16 05:19:33,482 - INFO - flashinfer.jit: Loading JIT ops: batch_prefill_with_kv_cache_dtype_q_f16_dtype_kv_e4m3_dtype_o_f16_dtype_idx_i32_he

... (13 outputs left out)

Capturing CUDA graph shapes:  80%|████████  | 4/5 [00:59<00:08,  8.53s/it]

WARNING 03-16 05:20:33 config.py:3486] Current VLLM config is not set.
(VllmWorkerProcess pid=351) (VllmWorkerProcess pid=359) (VllmWorkerProcess pid=354) WARNING 03-16 05:20:33 config.py:3486] Current VLLM config is not set.
WARNING 03-16 05:20:33 config.py:3486] Current VLLM config is not set.
WARNING 03-16 05:20:33 config.py:3486] Current VLLM config is not set.


Capturing CUDA graph shapes: 100%|██████████| 5/5 [01:01<00:00, 12.22s/it]

INFO 03-16 05:20:34 model_runner.py:1562] Graph capturing finished in 61 secs, took 0.14 GiB
(VllmWorkerProcess pid=351) (VllmWorkerProcess pid=359) INFO 03-16 05:20:34 model_runner.py:1562] Graph capturing finished in 61 secs, took 0.14 GiB
(VllmWorkerProcess pid=354) INFO 03-16 05:20:34 model_runner.py:1562] Graph capturing finished in 61 secs, took 0.14 GiB
INFO 03-16 05:20:34 model_runner.py:1562] Graph capturing finished in 61 secs, took 0.14 GiB
INFO 03-16 05:20:34 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 86.98 seconds


In [5]:
def extract_boxed_text(text):
    pattern = r'oxed{(.*?)}'
    matches = re.findall(pattern, text)
    if not matches:
        return ""
    for match in matches[::-1]:
        if match != "":
            return match
    return ""

def batch_message_filter(list_of_messages) -> tuple[list[list[dict]], list[str]]:
    extracted_answers = []
    list_of_messages_to_keep = []
    for messages in list_of_messages:
        answer = extract_boxed_text(messages[-1]['content'])
        if answer:
            extracted_answers.append(answer)
        else:
            list_of_messages_to_keep.append(messages)
    return list_of_messages_to_keep, extracted_answers

def select_answer(answers):
    counter = Counter()
    for answer in answers:
        try:
            if int(answer) == float(answer):
                if int(answer) == 0:
                    continue #skip 0 and all multiples of 1000
                counter[int(answer)] += 1 + random.random() / 1_000
        except:
            pass
    if not counter:
        return 210
    _, answer = sorted([(v,k) for k,v in counter.items()], reverse=True)[0]
    return answer%1000

def batch_message_generate(list_of_messages) -> list[list[dict]]:
    max_tokens =MAX_MODEL_LEN
    if (len(cutoff_times)>1) and (time.time() > cutoff_times[-2]):
        print("Speedrun")
        max_tokens = 2 * MAX_MODEL_LEN // 3
    sampling_params = SamplingParams(
        temperature=1.0,               # Randomness of the sampling
        #top_k=50,
        top_p=0.90,                    # Cumulative probability of the top tokens to consider
        min_p=0.05,                    # Minimum probability for a token to be considered
        skip_special_tokens=True,      # Whether to skip special tokens in the output
        max_tokens=max_tokens,         # Maximum number of tokens to generate
        stop=["</think>"],             # List of strings that stop the generation
        seed=777,
    )
    
    list_of_texts = [
        tokenizer.apply_chat_template(
            conversation=messages,
            tokenize=False,
            add_generation_prompt=True
        )
        for messages in list_of_messages
    ]

    request_output = llm.generate(
        prompts=list_of_texts,
        sampling_params=sampling_params,
    )
    print([len(single_request_output.outputs[0].token_ids) for single_request_output in request_output])

    sort_keys_and_list_of_messages = []
    for messages, single_request_output in zip(list_of_messages, request_output):
        #print()
        #print(single_request_output.outputs[0].text)
        #print()
        messages.append({'role': 'assistant', 'content': single_request_output.outputs[0].text})

        sort_keys_and_list_of_messages.append(
            (
                len(single_request_output.outputs[0].token_ids),
                messages
            )
        )
    print([sort_key for sort_key, _ in sort_keys_and_list_of_messages])
    sort_keys_and_list_of_messages.sort(key=lambda sort_key_and_messages: sort_key_and_messages[0])
    print([sort_key for sort_key, _ in sort_keys_and_list_of_messages])
    
    list_of_messages = [messages for _, messages in sort_keys_and_list_of_messages]
    return list_of_messages

In [6]:
def create_starter_messages(question, index):
    options = []
    for _ in range(13):
        options.append(
            [
                {"role": "system", "content": "A conversation between User and Assistant. The user asks a question, and the Assistant solves it.  The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. Please solve the following question step by step, think longer, and place your answer into answer within \\boxed{} ."},
                {"role": "user", "content": question },
            ]
        )
    for _ in range(3):    
        options.append(
            [
                {"role": "system", "content": "A conversation between User and Assistant. The user asks a question, and the Assistant solves it.  The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. Please solve the following question step by step, think longer, and place your answer into answer within \\boxed{} ."},
                {"role": "user", "content": question },
            ],
        )
    return options[index%len(options)]

def predict_for_question(question: str) -> int:
    #skip ASY
    if "\[asy\]" in question:
        cutoff_times.pop()
        return 3
    if "[asy]" in question:
        cutoff_times.pop()
        return 3
    selected_questions_only = True
    #selected_questions_only = False
    if selected_questions_only and not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        #if "Triangle" not in question:
        return 210 #speedup...
        #if "Triangle" not in question and "delightful" not in question and "George" not in question:
        #    return 210

    if time.time() > cutoff_time:
        return 3
    
    print(question)

    num_seqs = MAX_NUM_SEQS
    if time.time() > cutoff_times[-1]:
        num_seqs = 2 *  MAX_NUM_SEQS // 3
    list_of_messages = [create_starter_messages(question, index) for index in range(num_seqs)]

    all_extracted_answers = []
    for _ in range(1):
        list_of_messages = batch_message_generate(list_of_messages)
        
        if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            df = pd.DataFrame(
                {
                    "question": [question] * len(list_of_messages),
                    "message": [messages[-1]["content"] for messages in list_of_messages],
                }
            )
            df.to_csv(f"{str(int(time.time() - start_time)).zfill(5)}.csv", index=False)
        
        list_of_messages, extracted_answers = batch_message_filter(list_of_messages)
        all_extracted_answers.extend(extracted_answers)
    
    print(all_extracted_answers)
    answer = select_answer(all_extracted_answers)
    print(answer)

    print("\n\n")
    cutoff_times.pop()
    return answer

def predict(id_: pl.DataFrame, question: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
    id_ = id_.item(0)
    print("------")
    print(id_)
    question = question.item(0)
    answer = predict_for_question(question)
    print(question)
    print("------\n\n")
    return pl.DataFrame({'id': id_, 'answer': answer})

In [7]:
#predict_for_question("Triangle $ABC$ has side length $AB = 120$ and circumradius $R = 100$. Let $D$ be the foot of the perpendicular from $C$ to the line $AB$. What is the greatest possible length of segment $CD$?")

In [8]:
#predict_for_question("We call a sequence $a_1, a_2, \ldots$ of non-negative integers \textit{delightful} if there exists a positive integer $N$ such that for all $n > N$, $a_n = 0$, and for all $i \geq 1$, $a_i$ counts the number of multiples of $i$ in $a_1, a_2, \ldots, a_N$. How many delightful sequences of non-negative integers are there?")

In [9]:
#predict_for_question("For a positive integer $n$, let $S(n)$ denote the sum of the digits of $n$ in base 10. Compute $S(S(1)+S(2)+\cdots+S(N))$ with $N=10^{100}-2$.")

In [10]:
pd.read_csv(
    '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/reference.csv'
).drop('answer', axis=1).to_csv('reference.csv', index=False)

In [11]:

inference_server = kaggle_evaluation.aimo_2_inference_server.AIMO2InferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        (
#            '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/test.csv',
            'reference.csv',
        )
    )


------
480182
Let $ABC$ be a triangle with $BC=108$, $CA=126$, and $AB=39$. Point $X$ lies on segment $AC$ such that $BX$ bisects $\angle CBA$. Let $\omega$ be the circumcircle of triangle $ABX$. Let $Y$ be a point on $\omega$ different from $X$ such that $CX=CY$. Line $XY$ meets $BC$ at $E$. The length of the segment $BE$ can be written as $\frac{m}{n}$, where $m$ and $n$ are coprime positive integers. Find $m+n$.
------


------
192e23
Fred and George take part in a tennis tournament with $4046$ other players. In each round, the players are paired into $2024$ matches. How many ways are there to arrange the first round such that Fred and George do not have to play each other? (Two arrangements for the first round are \textit{different} if there is a player with a different opponent in the two arrangements.)
------


------
a1d40b
The Fibonacci numbers are defined as follows: $F_0 = 0$, $F_1 = 1$, and $F_{n+1} = F_n + F_{n-1}$ for $n \geq 1$. There are $N$ positive integers $n$ strictl